### 实验前的环境准备

这一格不是正式建模代码，而是实验开始前的“环境体检”。

为什么要先运行它？

1. 同一份 notebook 换一台电脑、换一个 Python 环境后，常见问题不是算法写错，而是缺少库。
2. 如果一开始不先检查，后面往往会在 `import`、画图、读文件或训练模型时突然报错，初学者很难判断问题到底出在代码还是环境。
3. 现在这一格已经升级为“先检查、再自动安装”，目的就是把环境问题尽量提前解决。

运行后你会看到几类信息：

1. 当前 notebook 实际使用的是哪个 Python 解释器。
2. 已经检测到哪些核心库。
3. 如果有缺失库，系统会尝试自动安装。
4. 如果安装完成后仍未生效，通常只需要重启内核，再从第 1 格重新运行。

可以把这一格理解成：正式做实验前，先把工具箱点一遍，缺什么先补什么。这样后面的每一步更容易顺利完成，也更符合真实开发中的工作流程。

In [ ]:
import importlib
import subprocess
import sys

required_packages = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'torch': 'torch',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'plotly': 'plotly',
    'nbformat': 'nbformat',
    'tqdm': 'tqdm',
    'openpyxl': 'openpyxl',
}

note_lines = [
    '说明：本教程需要读取 xlsx 文件，因此需要 openpyxl。',
    '说明：本教程使用 Plotly 在 notebook 中显示三维图，因此建议同时具备 nbformat。',
]


def is_module_available(module_name):
    return importlib.util.find_spec(module_name) is not None


installed_modules = []
missing_packages = []
for module_name, package_name in required_packages.items():
    if is_module_available(module_name):
        installed_modules.append(module_name)
    else:
        missing_packages.append(package_name)

print('当前 Python 解释器:', sys.executable)
print('已检测到的模块:', ', '.join(installed_modules) if installed_modules else '无')

if missing_packages:
    print('\n检测到缺失依赖，开始自动安装:')
    print('pip install ' + ' '.join(missing_packages))
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
        importlib.invalidate_caches()
        still_missing = [
            package_name
            for module_name, package_name in required_packages.items()
            if not is_module_available(module_name)
        ]
        if still_missing:
            print('\n以下依赖安装后仍未检测到，请重启内核后重试:')
            print(', '.join(still_missing))
        else:
            print('\n依赖已自动安装完成，可以继续运行本教程。')
    except subprocess.CalledProcessError as error:
        print(f'\n自动安装失败，返回码: {error.returncode}')
        print('请手动执行:')
        print('pip install ' + ' '.join(missing_packages))
else:
    print('\n依赖检查通过，可以继续运行本教程。')

for note in note_lines:
    print(note)

print('如果你已经安装过，但这里仍显示缺失，通常是因为当前 notebook 内核和安装库使用的 Python 环境不是同一个。')
print('如果刚完成自动安装，后续单元格仍报导入错误，重启内核后再从头运行一次。')

### 常规房价回归实践

本实验保留原来的房价预测主题和本地 Excel 数据读取方式，同时把讲解结构调整为与医疗回归教程一致的节奏：先检查原始表，再完成文本解析、特征工程、标准化与张量化，最后再进行 PyTorch 回归建模与结果分析。

- 数据集：某城市二手房源公开整理数据
- 本地文件：house.xlsx
- 任务：根据面积、区域、楼层、朝向、建筑时间、户型等信息预测房屋总价
- 数据集简介：该数据集保留了真实房源表中常见的文本字段、混合格式字段和价格信息，适合讲解“如何把生活语言整理成模型输入”
- 优点：场景直观、字段类型丰富、与医疗回归教程共享相同建模框架，便于横向比较不同任务

#### 本实验建议关注 4 个问题

1. 面积、楼层、建筑时间、户型这些文本字段是怎样一步步转成数值特征的？
2. 为什么表里虽然有单价，却不适合直接拿来预测总价？
3. 如果修改学习率、训练轮数或隐藏层宽度，回归结果会怎样变化？
4. 哪些价格区间更容易出现较大误差，为什么高价样本通常更难预测？

### 开始前先建立回归概念框架

为了让常规版和医疗版的回归教程能直接对照，先把这两个实验共享的几个核心概念说清楚：

1. 回归任务预测的是连续数值，本章里分别是医疗费用和房屋总价，而不是“属于哪一类”。
2. MLP 可以理解成一组不断做加权计算的神经元，隐藏层负责从多个字段里提炼组合规律。
3. 回归模型常用 `MSELoss()` 来衡量“预测值离真实值有多远”，误差越大，模型就越需要继续调整参数。
4. `MAE` 反映平均误差大小，`R²` 反映模型解释整体变化趋势的能力，这两个指标要结合着看。
5. `learning_rate` 决定每次参数更新迈多大步，`epochs` 决定模型反复练习多少轮；训练过少可能欠拟合，训练过头也可能出现过拟合。

先建立这套概念框架，后面再看代码，就更容易把“参数怎么调”和“模型为什么这样变”对应起来。

### 第一步：准备实验工具

这一步像是在厨房里把锅、刀、砧板都摆好。

后面要完成房价预测，需要几类工具协同工作：

1. `pandas`、`numpy`：负责整理房源表格数据。
2. `torch`：负责搭建和训练神经网络。
3. `sklearn`：负责标准化、划分训练集测试集和评价回归效果。
4. `matplotlib`、`plotly`：帮助我们观察分布、拟合效果和误差情况。

初学时不用急着记库名，更重要的是先知道：后面每段代码都不是“凭空出现”，而是在调用这里准备好的工具。

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
from matplotlib import font_manager
import plotly.express as px
import plotly.io as pio
from tqdm import trange

def configure_matplotlib_for_cjk():
    preferred_fonts = [
        "PingFang SC",
        "Hiragino Sans GB",
        "Heiti SC",
        "STHeiti",
        "Songti SC",
        "Arial Unicode MS",
        "Microsoft YaHei",
        "SimHei",
        "Noto Sans CJK SC",
        "Source Han Sans SC",
        "WenQuanYi Zen Hei",
        "DejaVu Sans",
    ]
    available_fonts = {font.name for font in font_manager.fontManager.ttflist}
    for font_name in preferred_fonts:
        if font_name in available_fonts:
            plt.rcParams["font.family"] = font_name
            break
    plt.rcParams["axes.unicode_minus"] = False

configure_matplotlib_for_cjk()
pio.renderers.default = "plotly_mimetype"

### 第二步：认识原始数据

房价数据往往不是“干净的数字矩阵”，里面常常混着文本描述、单位、地区信息和不规范写法。

所以这一步的重点不是立刻预测，而是先把原始房源信息拆清楚：

1. 哪些字段可以直接转成数字？
2. 哪些字段需要从文本里提取信息？
3. 目标值是哪一列？
4. 是否存在异常值或明显不合理记录？

可以把这一步理解成“先把中介给你的原始房源资料整理成统一格式”，否则模型根本不知道该看哪里。

### 原始特征与补充特征含义

房价数据的难点在于：很多字段不是标准数字，而是带单位、带文字、甚至带生活语义的描述。先把它们的现实含义讲清楚，学生后面就更容易理解为什么要做文本提取、编码和衍生特征构造。

**原始特征含义：**

1. `面积`：房屋建筑面积，通常是影响总价的核心因素。
2. `区域`：房屋所处区域，不同区域往往对应不同地段价值。
3. `楼层`：所在楼层，可能影响采光、便利性和价格。
4. `朝向`：房屋朝向，通常与采光和居住体验相关。
5. `建筑时间`：建成年份，可帮助判断房屋新旧程度。
6. `户型`：房屋格局描述，例如几室几厅。
7. `室`、`厅`：从户型字段中拆解出的房间数和客厅数。
8. `价格`：房屋总价，是本任务要预测的目标值。
9. `单价`：单位面积价格，虽然表中存在，但与总价关系过强，本教程不用它直接做输入，避免目标泄漏。

**补充特征含义：**

1. `房龄`：当前年份减去建成年份，帮助模型理解“新房/老房”的差异。
2. `总居室数`：室数加厅数，概括整体空间结构。
3. `每室面积`：面积除以室数，用来观察房屋是否“宽敞”。
4. `楼层面积交互`：面积与楼层的组合特征，用来描述“大面积且楼层不同”可能带来的联合影响。

可以提醒学生：这些补充特征并不是凭空捏造，而是把原本分散在文本或多个字段里的信息重新整理成更适合建模的数值表达。

### 第三步：构造衍生特征并完成编码

这一步和医疗回归教程的角色是一致的，都是把“人能看懂的数据”翻译成“模型能计算的输入”。

1. 原始房价表里很多字段带有文字、单位或混合格式，例如面积、楼层、建筑时间、户型，必须先解析成数字。
2. 通过构造 `房龄`、`总居室数`、`每室面积`、`楼层面积交互` 等衍生特征，可以让模型更容易看到空间结构和房屋新旧程度。
3. `区域`、`朝向` 这类类别信息也要先转成数值表达，模型才能继续计算。
4. 这一步不是机械地“清洗表格”，而是在把生活语义重新整理成更适合回归建模的特征结构。

**总结**：回归模型能不能学到规律，很大程度上取决于前面这一步是否把原始字段整理成了可解释、可计算、可比较的输入。

In [ ]:
# 读取数据，并进行预处理与特征工程
df = pd.read_excel('house.xlsx')

print("原始样本数, 原始字段数:", df.shape)
print("\n原始字段信息:")
print(df.info())
print("\n原始缺失值统计:")
print(df.isnull().sum())
print("\n原始重复行数量:", df.duplicated().sum())

df1 = df.drop_duplicates().dropna().copy()

df1['面积'] = pd.to_numeric(df1['面积'].str.extract(r'(\d+\.?\d+)', expand=False))
df1['价格'] = pd.to_numeric(df1['价格'].str.extract(r'(\d+)', expand=False))
df1['单价'] = pd.to_numeric(df1['单价'].str.extract(r'(\d+)', expand=False))
df1['楼层'] = pd.to_numeric(df1['楼层'].str.extract(r'(\d+)', expand=False))
df1['建筑时间'] = pd.to_numeric(df1['建筑时间'].str.replace('年建', '', regex=False))
df1['朝向'] = df1['朝向'].str.replace('朝', '', regex=False)
df1['朝向'] = df1['朝向'].str.replace('(进门) ', '', regex=False)
df1['朝向'] = df1['朝向'].str.replace('(进门)', '', regex=False)
df1 = df1[df1['朝向'] != ''].copy()
df1['朝向'] = pd.factorize(df1['朝向'])[0] + 1
df1['区域'] = pd.factorize(df1['区域'])[0] + 1
df1[['户型', '室', '厅']] = df1['户型'].str.extract(r'((\d+)室(\d+)厅)', expand=True)
df1['室'] = df1['室'].astype('int')
df1['厅'] = df1['厅'].astype('int')

df1['房龄'] = (pd.Timestamp.now().year - df1['建筑时间']).clip(lower=0)
df1['总居室数'] = df1['室'] + df1['厅']
df1['每室面积'] = (df1['面积'] / df1['室'].clip(lower=1)).round(2)
df1['楼层面积交互'] = (df1['面积'] * df1['楼层']).round(2) # 楼层面积交互特征，可以帮助模型捕捉楼层和面积之间的非线性关系

print("\n清洗后样本数, 字段数:", df1.shape)
print("\n建模前预览:")
print(df1[['面积', '区域', '楼层', '朝向', '建筑时间', '房龄', '室', '厅', '总居室数', '每室面积', '价格']].head())
print("\n教学提示：原始表中的文本字段必须经过提取、编码和类型转换，才能用于神经网络建模。")
print("教学提示：这里额外构造了房龄、总居室数、每室面积等衍生特征，让流程和医学回归教程保持一致。")
print("教学提示：虽然表里有单价，但它和总价高度相关，容易造成目标泄漏，因此本教程不把单价直接作为输入特征。")

df_model = df1[['面积', '区域', '楼层', '朝向', '建筑时间', '房龄', '室', '厅', '总居室数', '每室面积', '楼层面积交互', '价格']]

### 第四步：完成标准化、数据划分与张量化

这一格和医疗回归教程保持同样的操作顺序，都是先把特征拉到相近尺度，再把数据切成训练集和测试集，最后转成 PyTorch 张量。

1. **标准化**：面积、楼层、房龄等特征量纲差异很大，标准化后训练更稳定。
2. **训练集 / 测试集划分**：训练集负责让模型学习规律，测试集负责检查模型是不是只记住了样本。
3. **张量化**：PyTorch 训练依赖张量数据结构，后面的前向计算、误差回传和参数更新都建立在这里。

可以把这一步理解成：先把不同来源、不同量纲的房源信息整理到同一坐标系里，再交给模型开始正式学习。

In [ ]:
# 转化成张量并划分训练集和测试集
X = df_model.drop('价格', axis=1).astype(float)
y = df_model[['价格']].values

scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

print("训练集形状:", X_train.shape, y_train.shape)
print("测试集形状:", X_test.shape, y_test.shape)
print("\n教学提示：标准化后，不同量纲的特征更容易被网络同时学习。")
print("教学提示：这里把 X 保留成带列名的表结构，后面做特征重要性验证时更方便解释。")

#### 特征降维可视化

 > 先把多个特征压缩到三维空间，再观察样本在低维空间中的分布情况。

 > 现在改为三维交互式散点图，你可以在 notebook 输出中直接旋转、缩放和悬停观察不同价格区间样本的位置关系。

 > 如果不同价格区间在三维投影上仍大量重叠，则说明后续还可以继续思考更有效的特征工程。

In [ ]:
# 用 PCA 把多维特征压缩到三维空间，观察不同价格区间的样本分布
price_bins = pd.qcut(df_model['价格'], q=4, duplicates='drop')
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    '主成分1': X_pca[:, 0],
    '主成分2': X_pca[:, 1],
    '主成分3': X_pca[:, 2],
    '真实价格区间': price_bins.astype(str),
    '真实房价': df_model['价格'],
    '面积': df_model['面积'].round(2),
    '区域': df_model['区域'],
    '建筑时间': df_model['建筑时间'],
})

fig = px.scatter_3d(
    pca_df,
    x='主成分1',
    y='主成分2',
    z='主成分3',
    color='真实价格区间',
    hover_data=['真实房价', '面积', '区域', '建筑时间'],
    title='房价任务：特征降维后的三维交互分布',
    opacity=0.72,
    color_discrete_sequence=px.colors.qualitative.Set2,
 )
fig.update_traces(marker=dict(size=4))
fig.update_layout(margin=dict(l=0, r=0, t=50, b=0))
fig.show()

explained_ratio = pca.explained_variance_ratio_
print('前三个主成分的方差解释率:', np.round(explained_ratio, 4))
print('累计解释率:', round(float(explained_ratio.sum()), 4))
print('教学提示：你可以旋转图形，观察高价样本是否在三维空间里形成相对集中的区域。')

In [ ]:
# 学生可在这里修改训练参数，再重新运行后续单元
student_config = {
    "hidden_dims": [32, 16],
    "learning_rate": 0.001,
    "epochs": 1000,
}

print("当前实验参数:", student_config)
print("说明：根据小范围参数搜索，[32, 16] 在当前房价回归数据上比 [64, 32] 更稳，因此作为新的默认起点。")

#### 可调参数实验

下面这个单元把一部分关键参数单独拿出来，方便同学们边改边看结果。

当前默认结构已经根据小范围参数搜索调整为更稳的 `[32, 16]`，建议优先尝试：
- 把 epochs 从 1000 改成 600 或 1400
- 把 learning_rate 从 0.001 改成 0.0005 或 0.002
- 把 hidden_dims 从 [32, 16] 改回 [64, 32]，对比更宽网络是否真的更适合当前数据

观察 MAE、R² 和高价区间误差会怎样变化。

### 第五步：开始训练 MLP 回归模型

完成数据准备后，模型才真正开始学习“哪些房源特征会把价格往上拉，哪些特征会把价格往下拉”。

这里可以把训练过程理解成两件事：

1. 先定义网络结构：输入层接收房源特征，隐藏层逐步提炼组合模式，输出层给出预测总价。
2. 再反复执行训练循环：模型先预测，再用损失函数衡量误差，然后根据误差反向调整参数。

如果把它想得更形象一点，就是模型先给房子估价，老师告诉它偏高还是偏低，它再一轮轮修正，最后才会越来越接近真实价格。

In [ ]:
# 定义模型，损失函数和优化器
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims):
        super().__init__()
        layers = []
        previous_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(previous_dim, hidden_dim))  # 添加线性层
            layers.append(nn.ReLU())  # 添加ReLU激活函数
            previous_dim = hidden_dim  # 更新维度为当前隐藏层维度
        layers.append(nn.Linear(previous_dim, 1))  # 添加输出层，输出维度为1
        self.model = nn.Sequential(*layers)  # 将所有层组合成一个序列化的模型

    def forward(self, x):
        return self.model(x)  # 定义前向传播过程，输入x经过模型计算得到输出

hidden_dims = student_config.get("hidden_dims", [64, 32])
learning_rate = student_config.get("learning_rate", 0.001)
epochs = student_config.get("epochs", 1000)

mlp = MLP(X_train.shape[1], hidden_dims)
criterion = nn.MSELoss()
optimizer = optim.Adam(mlp.parameters(), lr=learning_rate)

In [ ]:
# 训练并验证模型
for epoch in trange(epochs, desc="Training Epochs"):
    mlp.train()
    optimizer.zero_grad()
    output = mlp(X_train)
    loss = criterion(output, y_train)
    loss.backward()
    optimizer.step()

mlp.eval()
with torch.no_grad():
    pred = mlp(X_test)
    test_loss = criterion(pred, y_test)
    print(f'Test Loss: {test_loss.item():.4f}')

    y_pred = scaler_y.inverse_transform(pred.numpy()).reshape(-1)
    y_true = scaler_y.inverse_transform(y_test.numpy()).reshape(-1)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"MAE: {mae:.2f}")
    print(f"R²: {r2:.3f}")
    print(f"当前参数: hidden_dims={hidden_dims}, learning_rate={learning_rate}, epochs={epochs}")

    plt.figure(figsize=(9, 5))
    plt.plot(range(len(y_pred)), y_pred, alpha=0.7, color='r', label='预测房价')
    plt.plot(range(len(y_pred)), y_true, alpha=0.7, color='b', label='真实房价')
    plt.xlabel('测试集样本序号')
    plt.ylabel('房价（万元）')
    plt.title('真实房价与预测房价对比')
    plt.legend()
    plt.tight_layout()
    plt.show()

    print("\n思考：如果模型曲线明显滞后于真实曲线，可能意味着模型容量不足或特征信息不够。")
    print("你可以尝试修改 student_config 中的参数，再重新运行本单元观察变化。")

### 第六步：分析测试集误差分布

回归任务里，只看一个平均误差往往不够。

更值得追问的是：

1. 模型在高价房和低价房上的误差是否一样？
2. 有没有某些价格区间明显更难预测？
3. 误差是普遍偏大，还是只集中在少数样本上？

这就像看考试成绩时，不只是看总分，还要看学生究竟是“整体都一般”，还是“某一类题特别容易失分”。

In [ ]:
# 误差分布可视化：按测试集真实价格区间统计误差范围
abs_error = np.abs(y_true - y_pred)
correlation = float(np.corrcoef(y_true, y_pred)[0, 1])

error_df = pd.DataFrame({
    '真实房价': y_true,
    '预测房价': y_pred,
    '绝对误差': abs_error,
})
error_df['价格区间'] = pd.qcut(
    error_df['真实房价'],
    q=4,
    duplicates='drop',
    precision=1,
 )

error_summary = error_df.groupby('价格区间', observed=False)['绝对误差'].agg(
    ['count', 'mean', 'median', lambda values: np.percentile(values, 90), 'max']
)
error_summary.columns = ['样本数', '平均误差', '中位误差', '90分位误差', '最大误差']
error_summary = error_summary.round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_true, y_pred, alpha=0.65, color='#2f7d6d', label='测试样本')
line_min = min(y_true.min(), y_pred.min())
line_max = max(y_true.max(), y_pred.max())
axes[0].plot([line_min, line_max], [line_min, line_max], 'r--', label='理想预测线')
axes[0].set_xlabel('真实房价（万元）')
axes[0].set_ylabel('预测房价（万元）')
axes[0].set_title('真实房价与预测房价的相关性')
axes[0].legend()

boxplot_data = [
    error_df.loc[error_df['价格区间'] == interval_label, '绝对误差'].values
    for interval_label in error_summary.index
 ]
axes[1].boxplot(boxplot_data, patch_artist=True)
axes[1].set_xticks(range(1, len(error_summary.index) + 1))
axes[1].set_xticklabels([str(label) for label in error_summary.index])
axes[1].set_xlabel('测试集真实价格区间')
axes[1].set_ylabel('绝对误差（万元）')
axes[1].set_title('不同价格区间的测试误差分布')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

print(f'真实价格与预测价格的相关系数: {correlation:.3f}')
print('\n不同价格区间的测试误差统计：')
print(error_summary)
print('\n教学提示：如果高价区间的平均误差、90 分位误差和最大误差明显更大，说明模型在高价样本上的稳定性更弱。')

In [ ]:
# 用置乱测试做一个适合初学者理解的特征重要性验证
feature_names = list(X.columns)
baseline_mae = float(mae)
X_test_np = X_test.numpy().copy()
y_true = scaler_y.inverse_transform(y_test.numpy()).reshape(-1)
importance_rows = []

for feature_index, feature_name in enumerate(feature_names):
    shuffled_X_test = X_test_np.copy()
    np.random.seed(42)
    np.random.shuffle(shuffled_X_test[:, feature_index])
    shuffled_tensor = torch.tensor(shuffled_X_test, dtype=torch.float32)

    with torch.no_grad():
        shuffled_pred_scaled = mlp(shuffled_tensor)
        shuffled_pred = scaler_y.inverse_transform(shuffled_pred_scaled.numpy()).reshape(-1)

    shuffled_mae = mean_absolute_error(y_true, shuffled_pred)
    importance_rows.append({
        '特征': feature_name,
        '打乱后MAE': round(float(shuffled_mae), 2),
        'MAE增加值': round(float(shuffled_mae - baseline_mae), 2),
    })

importance_df = pd.DataFrame(importance_rows).sort_values('MAE增加值', ascending=False)
print('基线 MAE:', round(baseline_mae, 2))
print('\n特征重要性验证结果：')
print(importance_df)

plt.figure(figsize=(8, 5))
plt.barh(importance_df['特征'], importance_df['MAE增加值'], color='#4c956c')
plt.gca().invert_yaxis()
plt.xlabel('打乱该特征后增加的 MAE')
plt.ylabel('特征')
plt.title('基于置乱测试的特征重要性验证')
plt.tight_layout()
plt.show()

print('教学提示：MAE 增加值越大，说明模型越依赖该特征。')
print('注意：这只是教学版的重要性验证，不等于严格的因果解释。')

In [ ]:
# 选做：小范围比较几组训练参数，判断默认配置是否需要调整
trial_configs = [
    {"hidden_dims": [64, 32], "learning_rate": 0.001, "epochs": 1000},
    {"hidden_dims": [64, 32], "learning_rate": 0.001, "epochs": 1400},
    {"hidden_dims": [32, 16], "learning_rate": 0.001, "epochs": 1000},
]

trial_results = []

for trial_config in trial_configs:
    torch.manual_seed(42)
    trial_mlp = MLP(X_train.shape[1], trial_config["hidden_dims"])
    trial_optimizer = optim.Adam(trial_mlp.parameters(), lr=trial_config["learning_rate"])
    trial_criterion = nn.MSELoss()

    for _ in trange(trial_config["epochs"], desc=f"Trial {trial_config}", leave=False):
        trial_mlp.train()
        trial_optimizer.zero_grad()
        trial_output = trial_mlp(X_train)
        trial_loss = trial_criterion(trial_output, y_train)
        trial_loss.backward()
        trial_optimizer.step()

    trial_mlp.eval()
    with torch.no_grad():
        trial_pred_scaled = trial_mlp(X_test)
        trial_pred_value = scaler_y.inverse_transform(trial_pred_scaled.numpy()).reshape(-1)
        trial_true_value = scaler_y.inverse_transform(y_test.numpy()).reshape(-1)

    trial_results.append({
        "hidden_dims": str(trial_config["hidden_dims"]),
        "learning_rate": trial_config["learning_rate"],
        "epochs": trial_config["epochs"],
        "MAE": round(mean_absolute_error(trial_true_value, trial_pred_value), 2),
        "R2": round(r2_score(trial_true_value, trial_pred_value), 4),
    })

trial_results_df = pd.DataFrame(trial_results).sort_values(["MAE", "R2"], ascending=[True, False])
print(trial_results_df)
print("\n说明：如果候选配置只是轻微波动，不足以明显改善解释性或稳定性，就保留当前默认参数。")

#### 特征重要性反思与验证

下面不是严格意义上的统计学因果分析，而是一个适合初学者的“特征重要性验证”思路：

- 先保留当前训练好的模型作为基线
- 然后每次只打乱一个特征在测试集中的取值顺序
- 如果打乱某个特征后误差明显变大，说明模型较依赖这个特征

#### 结果反思与动手挑战

请同学们结合本次运行结果尝试回答下面几个问题：

1. 本次运行里，房价回归的 $R^2$ 只有约 0.42，明显低于医学回归案例。为什么房价任务更难？
2. 从价格区间统计看，最高价格区间的平均误差和 90 分位误差都更大，这说明了什么？
3. 三维交互图的累计解释率已经超过 0.7，但模型效果仍然一般。为什么“看起来能分层”不代表“预测就一定很准”？
4. 置乱测试里，`总居室数`、`面积`、`室` 排在前面，而 `每室面积` 反而靠后。你如何解释这个结果？
5. 为什么新增的 `房龄`、`楼层面积交互` 会有一定作用，但没有超过面积和居室结构？
6. 如果把隐藏层调大或把 epochs 调大，结果一定会更好吗？还是更可能受到样本噪声和隐含因素限制？

<details>
<summary>提示与参考答案</summary>

提示：把三维交互图、真实价格与预测价格相关性图、价格区间误差统计和置乱重要性结果结合起来看。

参考答案：
1. 房价受位置、装修、学区、交通、政策、采光等许多隐含因素影响，当前字段只能解释其中一部分，所以这个任务比医学回归更难。
2. 高价区间样本更少、形成机制更复杂，所以平均误差和高分位误差更容易放大。
3. 三维投影能说明特征里存在一定结构，但它只是压缩后的可视化，不代表模型已经掌握了全部高维关系。
4. `总居室数`、`面积`、`室` 排在前面，说明房屋规模和户型结构对当前模型最关键；`每室面积` 靠后，说明它提供的是补充信息，而不是主导信息。
5. `房龄`、`楼层面积交互` 能帮助模型刻画新旧程度和空间结构，但通常不会比面积和户型更直接。
6. 更大的隐藏层和更多训练轮数不一定更好，因为当前误差的一部分可能来自表外因素，而不只是模型容量不足。
</details>